In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Загружаем данные...")
# 1. Загружаем наш подготовленный датасет
df = pd.read_csv('merged_toxicity_dataset.csv')

# Базовая очистка: удаляем пустые строки и приводим текст к строковому типу
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)

# Метки toxic переводим в целые числа (из 0.0 в 0)
df['toxic'] = df['toxic'].astype(int)

# 2. Разделение данных
# Мы откладываем 20% данных "в сейф" (test_size=0.2), чтобы в конце проверить модель 
# на текстах, которые она никогда раньше не видела.
X = df[['text']]
y = df['toxic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Строк для обучения: {len(X_train)}")
print(f"Строк для проверки: {len(X_test)}\n")

# 3. Настройка параметров CatBoost
model = CatBoostClassifier(
    iterations=500,           # Количество проходов (деревьев). 500 для старта — оптимально.
    learning_rate=0.1,        # Шаг обучения
    depth=6,                  # Глубина анализа контекста
    text_features=['text'],   # ГЛАВНАЯ ФИШКА: говорим алгоритму, что это текст!
    eval_metric='Accuracy',   # Метрика - точность
    task_type='CPU',          # Если у тебя крутая видеокарта NVIDIA, напиши тут 'GPU'
    verbose=100               # Писать лог каждые 100 шагов
)

# 4. Запускаем обучение!
print("Начинаем обучение модели. Это может занять несколько минут...")
model.fit(X_train, y_train, eval_set=(X_test, y_test))

# 5. Проверка качества модели (Тестирование)
print("\n--- ОЦЕНКА КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ ---")
# Просим модель предсказать токсичность для тех 20% данных из "сейфа"
y_pred = model.predict(X_test)
# Сравниваем её ответы с реальными
print(classification_report(y_test, y_pred))

# 6. Сохранение "мозга" модели
model.save_model('toxicity_model.cbm')
print("\nУра! Модель успешно сохранена в файл 'toxicity_model.cbm'")

Загружаем данные...
Строк для обучения: 139186
Строк для проверки: 34797

Начинаем обучение модели. Это может занять несколько минут...
0:	learn: 0.8910954	test: 0.8954795	best: 0.8954795 (0)	total: 262ms	remaining: 2m 10s
100:	learn: 0.9246476	test: 0.9246774	best: 0.9246774 (100)	total: 10.5s	remaining: 41.5s
200:	learn: 0.9286207	test: 0.9259994	best: 0.9260281 (177)	total: 20.4s	remaining: 30.4s
300:	learn: 0.9307330	test: 0.9270339	best: 0.9270339 (291)	total: 30.2s	remaining: 20s
400:	learn: 0.9324070	test: 0.9271776	best: 0.9275512 (373)	total: 40.1s	remaining: 9.9s
499:	learn: 0.9336356	test: 0.9272638	best: 0.9275512 (373)	total: 49.7s	remaining: 0us

bestTest = 0.9275512257
bestIteration = 373

Shrink model to first 374 iterations.

--- ОЦЕНКА КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ ---
              precision    recall  f1-score   support

           0       0.94      0.98      0.96     30546
           1       0.82      0.52      0.64      4251

    accuracy                           

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Загружаем данные...")
df = pd.read_csv('merged_toxicity_dataset.csv')
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)
df['toxic'] = df['toxic'].astype(int)

X = df[['text']]
y = df['toxic']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Инициализируем продвинутую модель...")
# Добавляем магию: auto_class_weights и смену метрики на F1
model = CatBoostClassifier(
    iterations=600, 
    text_features=['text'],
    eval_metric='F1',               # Фокус на поиске токсичных комментариев
    auto_class_weights='Balanced',  # Автоматический баланс 30k хороших против 4k плохих
    task_type='CPU',
    verbose=0                       # Отключаем спам в консоль при переборе
)

# Сетка параметров для перебора (Grid Search)
# Модель обучится 4 раза с разными настройками и выберет лучшую
grid = {
    'learning_rate': [0.1, 0.2],
    'depth': [4, 6]
}

print("Запускаем умный перебор параметров (Grid Search). Придется подождать...")
grid_search_result = model.grid_search(
    grid, 
    X=X_train, 
    y=y_train, 
    cv=3, # Кросс-валидация
    plot=True
)

print("\n--- ЛУЧШИЕ ПАРАМЕТРЫ ---")
print(grid_search_result['params'])

print("\n--- НОВАЯ ОЦЕНКА КАЧЕСТВА ---")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

model.save_model('toxicity_model_tuned.cbm')
print("Улучшенная модель сохранена как 'toxicity_model_tuned.cbm'!")

Загружаем данные...
Инициализируем продвинутую модель...
Запускаем умный перебор параметров (Grid Search). Придется подождать...


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))


bestTest = 0
bestIteration = 0

0:	loss: 0.0000000	best: 0.0000000 (0)	total: 10s	remaining: 30.1s

bestTest = 0
bestIteration = 0

1:	loss: 0.0000000	best: 0.0000000 (0)	total: 15.7s	remaining: 15.7s

bestTest = 0
bestIteration = 0

2:	loss: 0.0000000	best: 0.0000000 (0)	total: 21.6s	remaining: 7.19s

bestTest = 0
bestIteration = 0

3:	loss: 0.0000000	best: 0.0000000 (0)	total: 27.2s	remaining: 0us
Estimating final quality...
Training on fold [0/3]

bestTest = 0.854301793
bestIteration = 437

Training on fold [1/3]

bestTest = 0.860263133
bestIteration = 576

Training on fold [2/3]

bestTest = 0.8648863593
bestIteration = 539


--- ЛУЧШИЕ ПАРАМЕТРЫ ---
{'depth': 4, 'learning_rate': 0.1}

--- НОВАЯ ОЦЕНКА КАЧЕСТВА ---
              precision    recall  f1-score   support

           0       0.98      0.88      0.93     30546
           1       0.50      0.85      0.63      4251

    accuracy                           0.88     34797
   macro avg       0.74      0.86      0.78     34797